In [ ]:
import sys
sys.path.append('..')

import yaml
import datetime as dt
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore")

from src.strategies.bollinger import DynamicMeanReversionStrategy
from src.strategies.zscore import DynamicZScoreStrategy
from src.strategies.kalman import KalmanZScoreStrategy
from src.engine.backtest import PairsBacktest

# Load Settings
with open('../config/settings.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# SETUP DATE SPLIT (15 Calendar Years: 70% Train / 30% Test)
split_ratio = config['backtest']['split_ratio']
lookback_years = config['backtest']['lookback_years']
total_calendar_days = int(lookback_years * 365.25)

end_date = pd.to_datetime(config['backtest']['end_date']).date()
start_date = end_date - pd.Timedelta(days=total_calendar_days)
train_days = int(total_calendar_days * split_ratio)
end_date_train = start_date + pd.Timedelta(days=train_days)
start_date_test = end_date_train + pd.Timedelta(days=1)

t_cost = config['backtest']['transaction_cost']
is_compounding = config['backtest']['compounding']
metric_to_use = config['backtest']['target_metric']

strategy_name_BB = config['strategy_names']['bollinger']
strategy_name_ZS = config['strategy_names']['zscore']
strategy_name_KF = config['strategy_names']['kalman']

b3_pairs_universe = config['multi_pair']['universe']

print(f"[*] Optimization Target Metric set to: {metric_to_use.upper()}")
print(f"[*] Loaded {len(b3_pairs_universe)} pairs from universe.")


In [ ]:
# ==============================================================================
# BOLLINGER BANDS STRATEGY BATCH OPTIMIZATION
# ==============================================================================
windows_grid = config['grids']['bollinger']['ols_windows']
bollinger_param_grid = {
    'num_std': [round(x, 2) for x in np.linspace(1.0, 2.0, 15)],
    'stop_mult': config['grids']['bollinger']['stop_mult'],
    'ma_window': config['grids']['bollinger']['ma_window']
}

bollinger_train_results = {}
bollinger_test_results = {}
bollinger_best_params = {}

print(f"{'=' * 70}")
print("STARTING BATCH OPTIMIZATION - BOLLINGER BANDS STRATEGY")
print(f"{'=' * 70}\n")

for ticker_x, ticker_y, description in b3_pairs_universe:
    pair_label = f"{ticker_y} vs {ticker_x}"
    print(f"Processing: {pair_label:<25} | {description[:35]:<35} ...", end=" ", flush=True)

    try:
        backtest_all = PairsBacktest(
            ticker_x=ticker_x,
            ticker_y=ticker_y,
            start_date=str(start_date),
            end_date_train=str(end_date_train),
            start_date_test=str(start_date_test),
            end_date=str(end_date),
            strategy_cls=DynamicMeanReversionStrategy,
            strategy_param_grid=bollinger_param_grid,
            ols_windows_to_test=windows_grid,
            show_heatmaps=False, 
            transaction_cost=t_cost,
            compounding=is_compounding,
            target_metric=metric_to_use,
            strategy_name=strategy_name_BB
        )

        final_train_df, final_test_df = backtest_all.execute_pipeline()
        
        if final_train_df is not None and final_test_df is not None:
            bollinger_train_results[pair_label] = final_train_df
            bollinger_test_results[pair_label] = final_test_df
            bollinger_best_params[pair_label] = backtest_all.best_params
            print("✅ Done")
        else:
            print("⚠️ No results (None)")
    except Exception as e:
        print(f"❌ Error processing {pair_label}: {e}")

print(f"\n{'=' * 70}")
print("BATCH OPTIMIZATION FINISHED.")


In [ ]:
# ==============================================================================
# CONSOLIDATE METRICS AND VISUALIZE (OUT-OF-SAMPLE) - BOLLINGER BANDS
# ==============================================================================
summary_data = []

for pair_label, df in bollinger_test_results.items():
    if df is not None and not df.empty:
        if is_compounding:
            gross_return = ((df['strategy_returns'].fillna(0.0) + 1.0).cumprod().iloc[-1] - 1.0) * 100.0
            net_return = ((df['net_strategy_returns'].fillna(0.0) + 1.0).cumprod().iloc[-1] - 1.0) * 100.0
        else:
            gross_return = df['strategy_returns'].fillna(0.0).sum() * 100.0
            net_return = df['net_strategy_returns'].fillna(0.0).sum() * 100.0
            
        cost_drag = gross_return - net_return
        summary_data.append({
            'Pair': pair_label,
            'Gross Return (%)': round(gross_return, 2),
            'Transaction Cost (%)': round(cost_drag, 2),
            'Net Return (%)': round(net_return, 2)
        })

if summary_data:
    df_summary = pd.DataFrame(summary_data).sort_values(by='Net Return (%)', ascending=False).reset_index(drop=True)
    calculation_mode = "COMPOUNDING (Reinvested)" if is_compounding else "SIMPLE (Fixed Size)"
    print(f"\nPERFORMANCE SUMMARY TABLE (OUT-OF-SAMPLE) - {calculation_mode}:\n")
    print(df_summary.to_string(index=False))

    fig_bar = px.bar(
        df_summary, x='Pair', y='Net Return (%)',
        title=f'OOS Net Return Comparison per Pair ({strategy_name_BB})',
        color='Net Return (%)', color_continuous_scale='RdYlGn', text='Net Return (%)'
    )
    fig_bar.update_layout(xaxis_tickangle=-45, yaxis_title="Cumulative Net Return (%)", xaxis_title="Traded Pairs")
    fig_bar.show()

    fig_line = go.Figure()
    for pair_label, df in bollinger_test_results.items():
        if df is not None and not df.empty:
            cum_returns = ((df['net_strategy_returns'].fillna(0.0) + 1.0).cumprod() - 1.0) * 100.0 if is_compounding else df['net_strategy_returns'].fillna(0.0).cumsum() * 100.0
            fig_line.add_trace(go.Scatter(x=df.index, y=cum_returns, mode='lines', name=pair_label, opacity=0.7))

    fig_line.update_layout(
        title=f'Cumulative Net Return Evolution ({strategy_name_BB} - Test Period)',
        xaxis_title='Date', yaxis_title='Cumulative Return (%)', hovermode='x unified',
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=1.01)
    )
    fig_line.show()


In [ ]:
# ==============================================================================
# Z-SCORE STRATEGY BATCH OPTIMIZATION
# ==============================================================================
ols_windows_grid = config['grids']['zscore']['ols_windows']
zscore_param_grid = {
    'entry_z': [1, 1.25, 1.5, 1.75, 2.0, 2.25, 2.5],
    'exit_z': config['grids']['zscore']['exit_z'],
    'stop_loss_z': config['grids']['zscore']['stop_loss_z'],
    'lookback_window': config['grids']['zscore']['lookback_window']
}

zscore_train_results = {}
zscore_test_results = {}
zscore_best_params = {}

print(f"{'=' * 70}")
print("STARTING BATCH OPTIMIZATION - Z-SCORE STRATEGY")
print(f"{'=' * 70}\n")

for ticker_x, ticker_y, description in b3_pairs_universe:
    pair_label = f"{ticker_y} vs {ticker_x}"
    print(f"Processing: {pair_label:<25} | {description[:35]:<35} ...", end=" ", flush=True)

    try:
        zscore_backtest_all = PairsBacktest(
            ticker_x=ticker_x,
            ticker_y=ticker_y,
            start_date=str(start_date),
            end_date_train=str(end_date_train),
            start_date_test=str(start_date_test),
            end_date=str(end_date),
            strategy_cls=DynamicZScoreStrategy,
            strategy_param_grid=zscore_param_grid,
            ols_windows_to_test=ols_windows_grid,
            show_heatmaps=False, 
            transaction_cost=t_cost,
            compounding=is_compounding,
            target_metric=metric_to_use,
            strategy_name=strategy_name_ZS
        )

        zscore_train_df, zscore_test_df = zscore_backtest_all.execute_pipeline()
        
        if zscore_train_df is not None and zscore_test_df is not None:
            zscore_train_results[pair_label] = zscore_train_df
            zscore_test_results[pair_label] = zscore_test_df
            zscore_best_params[pair_label] = zscore_backtest_all.best_params
            print("✅ Done")
        else:
            print("⚠️ No results (None)")
    except Exception as e:
        print(f"❌ Error processing {pair_label}: {e}")

print(f"\n{'=' * 70}")
print("BATCH OPTIMIZATION FINISHED.")


In [ ]:
# ==============================================================================
# CONSOLIDATE METRICS AND VISUALIZE (OUT-OF-SAMPLE) - Z-SCORE
# ==============================================================================
summary_data = []

for pair_label, df in zscore_test_results.items():
    if df is not None and not df.empty:
        if is_compounding:
            gross_return = ((df['strategy_returns'].fillna(0.0) + 1.0).cumprod().iloc[-1] - 1.0) * 100.0
            net_return = ((df['net_strategy_returns'].fillna(0.0) + 1.0).cumprod().iloc[-1] - 1.0) * 100.0
        else:
            gross_return = df['strategy_returns'].fillna(0.0).sum() * 100.0
            net_return = df['net_strategy_returns'].fillna(0.0).sum() * 100.0
            
        cost_drag = gross_return - net_return
        summary_data.append({
            'Pair': pair_label,
            'Gross Return (%)': round(gross_return, 2),
            'Transaction Cost (%)': round(cost_drag, 2),
            'Net Return (%)': round(net_return, 2)
        })

if summary_data:
    df_summary = pd.DataFrame(summary_data).sort_values(by='Net Return (%)', ascending=False).reset_index(drop=True)
    calculation_mode = "COMPOUNDING (Reinvested)" if is_compounding else "SIMPLE (Fixed Size)"
    print(f"\nPERFORMANCE SUMMARY TABLE (OUT-OF-SAMPLE) - {calculation_mode}:\n")
    print(df_summary.to_string(index=False))

    fig_bar = px.bar(
        df_summary, x='Pair', y='Net Return (%)',
        title=f'OOS Net Return Comparison per Pair ({strategy_name_ZS})',
        color='Net Return (%)', color_continuous_scale='RdYlGn', text='Net Return (%)'
    )
    fig_bar.update_layout(xaxis_tickangle=-45, yaxis_title="Cumulative Net Return (%)", xaxis_title="Traded Pairs")
    fig_bar.show()

    fig_line = go.Figure()
    for pair_label, df in zscore_test_results.items():
        if df is not None and not df.empty:
            cum_returns = ((df['net_strategy_returns'].fillna(0.0) + 1.0).cumprod() - 1.0) * 100.0 if is_compounding else df['net_strategy_returns'].fillna(0.0).cumsum() * 100.0
            fig_line.add_trace(go.Scatter(x=df.index, y=cum_returns, mode='lines', name=pair_label, opacity=0.7))

    fig_line.update_layout(
        title=f'Cumulative Net Return Evolution ({strategy_name_ZS} - Test Period)',
        xaxis_title='Date', yaxis_title='Cumulative Return (%)', hovermode='x unified',
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=1.01)
    )
    fig_line.show()


In [ ]:
# ==============================================================================
# KALMAN FILTER STRATEGY BATCH OPTIMIZATION
# ==============================================================================
kalman_param_grid = {
    'entry_z': [1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2.5],
    'exit_z': config['grids']['kalman']['exit_z'],
    'stop_loss_z': config['grids']['kalman']['stop_loss_z'],
    'delta': config['grids']['kalman']['delta'],
    'obs_variance': config['grids']['kalman']['obs_variance']
}

kalman_train_results = {}
kalman_test_results = {}
kalman_best_params = {}

print(f"{'=' * 70}")
print("STARTING BATCH OPTIMIZATION - KALMAN FILTER STRATEGY")
print(f"{'=' * 70}\n")

for ticker_x, ticker_y, description in b3_pairs_universe:
    pair_label = f"{ticker_y} vs {ticker_x}"
    print(f"Processing: {pair_label:<25} | {description[:35]:<35} ...", end=" ", flush=True)

    try:
        kalman_backtest_all = PairsBacktest(
            ticker_x=ticker_x,
            ticker_y=ticker_y,
            start_date=str(start_date),
            end_date_train=str(end_date_train),
            start_date_test=str(start_date_test),
            end_date=str(end_date),
            strategy_cls=KalmanZScoreStrategy,
            strategy_param_grid=kalman_param_grid,
            show_heatmaps=False, 
            transaction_cost=t_cost,
            compounding=is_compounding,
            target_metric=metric_to_use,
            strategy_name=strategy_name_KF
        )

        kalman_train_df, kalman_test_df = kalman_backtest_all.execute_pipeline()
        
        if kalman_train_df is not None and kalman_test_df is not None:
            kalman_train_results[pair_label] = kalman_train_df
            kalman_test_results[pair_label] = kalman_test_df
            kalman_best_params[pair_label] = kalman_backtest_all.best_params
            print("✅ Done")
        else:
            print("⚠️ No results (None)")
    except Exception as e:
        print(f"❌ Error processing {pair_label}: {e}")

print(f"\n{'=' * 70}")
print("BATCH OPTIMIZATION FINISHED.")


In [ ]:
# ==============================================================================
# CONSOLIDATE METRICS AND VISUALIZE (OUT-OF-SAMPLE) - KALMAN FILTER
# ==============================================================================
summary_data = []

for pair_label, df in kalman_test_results.items():
    if df is not None and not df.empty:
        if is_compounding:
            gross_return = ((df['strategy_returns'].fillna(0.0) + 1.0).cumprod().iloc[-1] - 1.0) * 100.0
            net_return = ((df['net_strategy_returns'].fillna(0.0) + 1.0).cumprod().iloc[-1] - 1.0) * 100.0
        else:
            gross_return = df['strategy_returns'].fillna(0.0).sum() * 100.0
            net_return = df['net_strategy_returns'].fillna(0.0).sum() * 100.0
            
        cost_drag = gross_return - net_return
        summary_data.append({
            'Pair': pair_label,
            'Gross Return (%)': round(gross_return, 2),
            'Transaction Cost (%)': round(cost_drag, 2),
            'Net Return (%)': round(net_return, 2)
        })

if summary_data:
    df_summary = pd.DataFrame(summary_data).sort_values(by='Net Return (%)', ascending=False).reset_index(drop=True)
    calculation_mode = "COMPOUNDING (Reinvested)" if is_compounding else "SIMPLE (Fixed Size)"
    print(f"\nPERFORMANCE SUMMARY TABLE (OUT-OF-SAMPLE) - {calculation_mode}:\n")
    print(df_summary.to_string(index=False))

    fig_bar = px.bar(
        df_summary, x='Pair', y='Net Return (%)',
        title=f'OOS Net Return Comparison per Pair ({strategy_name_KF})',
        color='Net Return (%)', color_continuous_scale='RdYlGn', text='Net Return (%)'
    )
    fig_bar.update_layout(xaxis_tickangle=-45, yaxis_title="Cumulative Net Return (%)", xaxis_title="Traded Pairs")
    fig_bar.show()

    fig_line = go.Figure()
    for pair_label, df in kalman_test_results.items():
        if df is not None and not df.empty:
            cum_returns = ((df['net_strategy_returns'].fillna(0.0) + 1.0).cumprod() - 1.0) * 100.0 if is_compounding else df['net_strategy_returns'].fillna(0.0).cumsum() * 100.0
            fig_line.add_trace(go.Scatter(x=df.index, y=cum_returns, mode='lines', name=pair_label, opacity=0.7))

    fig_line.update_layout(
        title=f'Cumulative Net Return Evolution ({strategy_name_KF} - Test Period)',
        xaxis_title='Date', yaxis_title='Cumulative Return (%)', hovermode='x unified',
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=1.01)
    )
    fig_line.show()
